# PHP 리뷰 검색식의 정밀도

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/php_review_fixtures.ipynb)

제시된 두 검색 패턴으로 직접 작성한 PHP 단편을 정적으로 분류합니다. PHP를 실행하지 않습니다. 사례 이름은 용도와 공개 범위를 지정합니다. [검사 패턴 원본](https://github.com/No1Joon/oh-my-skills/blob/e9893ebc3f099b8e377286c743ef63a11b244d4a/plugins/security/skills/php-security-hardening/references/grep-patterns.md). 언어는 CHART_LANG=ko/en/both로 선택합니다.

In [ ]:
import os
import platform
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

language = os.environ.get("CHART_LANG", "ko")
if language not in {"ko", "en", "both"}:
    raise ValueError("CHART_LANG must be ko, en or both")
languages = ["ko", "en"] if language == "both" else [language]
if "ko" in languages:
    # Google Fonts, Nanum Gothic (SIL Open Font License 1.1).
    font = Path("NanumGothic-Regular.ttf")
    if not font.exists():
        urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf", font)
    font_manager.fontManager.addfont(str(font))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=font).get_name()
print("Python", platform.python_version(), "matplotlib", matplotlib.__version__)

In [ ]:
"""Measure two PHP review patterns against labeled, inert snippets. Never run PHP."""
import re
from dataclasses import dataclass

from pygments import lex
from pygments.lexers.web import PhpLexer
from pygments.token import Comment, String

WEAK_DIGEST = r"\b(md5|sha1)\s*\("
DEBUG_OUTPUT = r"\b(phpinfo|var_dump|var_export|print_r|debug_zval_dump)\s*\("


@dataclass(frozen=True)
class Example:
    name: str
    group: str
    needs_review: bool
    code: str


# Each label is part of the fixture contract. "Review" here means a digest
# used for a password, or diagnostic output sent by a public page.
CASES = [
    Example("password A", "digest", True, '$passwordDigest = md5($password);'),
    Example("password B", "digest", True, '$passwordDigest = sha1($password);'),
    Example("asset cache", "digest", False, '$assetCacheKey = md5($publicCss);'),
    Example("legacy checksum", "digest", False, '$legacyChecksum = sha1($publicFixture);'),
    Example("migration note", "digest", False, '// Remove md5() from password handling.'),
    Example("manual excerpt", "digest", False, '$manual = "sha1()";'),
    Example("password API", "digest", False, '$passwordDigest = password_hash($password, PASSWORD_DEFAULT);'),
    Example("public status page", "debug", True, 'phpinfo();'),
    Example("public profile page", "debug", True, 'var_dump($account);'),
    Example("public help page", "debug", True, 'print_r($internalSettings);'),
    Example("private CLI", "debug", False, 'var_dump($buildResult);'),
    Example("capture as text", "debug", False, '$exampleText = var_export($publicFixture, true);'),
    Example("comment", "debug", False, '/* Remove phpinfo() before shipping. */'),
    Example("documentation", "debug", False, '$manual = "print_r()";'),
    Example("public label", "debug", False, 'echo "Service status";'),
]


def mask_noncode(source):
    # Keep one placeholder per character so adjacent identifiers cannot join.
    return "".join(" " * len(value) if kind in Comment or kind in String else value
                   for kind, value in lex("<?php " + source, PhpLexer()))


def classify(case, tokens=False):
    pattern = WEAK_DIGEST if case.group == "digest" else DEBUG_OUTPUT
    source = mask_noncode(case.code) if tokens else case.code
    return bool(re.search(pattern, source))


def evaluate(tokens=False):
    counts = dict(TP=0, FP=0, TN=0, FN=0)
    for case in CASES:
        hit = classify(case, tokens)
        key = ("T" if hit == case.needs_review else "F") + ("P" if hit else "N")
        counts[key] += 1
    return counts

In [ ]:
import pygments
print("Pygments", pygments.__version__, "cases", len(CASES))
scores = {}
for name, tokens in [("text", False), ("tokens", True)]:
    values = evaluate(tokens)
    tp, fp, fn = (values[k] for k in ("TP", "FP", "FN"))
    scores[name] = values
    print(name, values, f"precision={tp/(tp+fp):.3f}", f"recall={tp/(tp+fn):.3f}")
for case in CASES:
    print(case.name, "label", int(case.needs_review), "text", int(classify(case)), "tokens", int(classify(case, True)))

In [ ]:
import numpy as np
for lang in languages:
    ko = lang == "ko"
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    x = np.arange(2)
    for i, key in enumerate(("TP", "FP")):
        values = [scores[mode][key] for mode in ("text", "tokens")]
        label = (["검토 대상 적중", "오탐"] if ko else ["True positives", "False positives"])[i]
        bars = ax.bar(x + (i - 0.5) * 0.34, values, 0.34, label=label, color=["#346853","#b56042"][i])
        ax.bar_label(bars, padding=3)
    ax.set_xticks(x, ["원문 검색", "주석과 문자열 제외"] if ko else ["Raw source", "Code tokens"])
    ax.set_ylabel("사례 수" if ko else "Snippets")
    ax.set_title("같은 PHP 예제 15개에서 찾은 후보" if ko else "Candidates found in the same 15 PHP snippets")
    ax.set_ylim(0, 10)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.18)
    ax.legend(frameon=False)
    plt.show()
    plt.close(fig)